# Experiment 13: Neural Machine Translation with Attention

**Aim:** Build a sequence-to-sequence (encoder-decoder) model with an **attention mechanism** that translates English sentences into Spanish.

**Dataset:** Anki/Tatoeba English-Spanish sentence pairs (`spa-eng`, ~118k pairs; the first 30,000 short pairs are used here).

**Architecture**
* **Encoder:** Embedding -> Bidirectional GRU -> sequence of hidden states (one per source word).
* **Decoder:** Embedding -> GRU initialised with the encoder summary state.
* **Attention (Luong dot-product):** for each decoder step, the decoder state is compared with all encoder states, the scores are soft-maxed into attention weights, and the weighted sum (*context vector*) is concatenated with the decoder state before predicting the next word.
* Training uses **teacher forcing**; inference decodes word-by-word (greedy search). Attention weights are plotted as alignment heat-maps.

In [ ]:
import numpy as np, re, unicodedata, zipfile, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(5); np.random.seed(5)
NUM_PAIRS, MAXLEN, UNITS, EMB, VOCAB_SIZE = 30000, 12, 256, 128, 6000
EPOCHS, BATCH = 12, 128
print("TensorFlow", tf.__version__)

## 1. Download and clean the data
Sentences are lower-cased, accents stripped from the English side only where harmless, punctuation separated from words, and wrapped in `<start>` / `<end>` tokens.

In [ ]:
zip_path = keras.utils.get_file("spa-eng.zip", "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip", extract=False)
import os
with zipfile.ZipFile(zip_path) as z: lines = z.read("spa-eng/spa.txt").decode("utf-8").strip().split("\n")

def norm(s):
    s = s.lower().strip()
    s = re.sub(r"([?.!,¿])", r" \1 ", s)
    s = re.sub(r"[^a-záéíóúñü?.!,¿]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

pairs = [(norm(l.split("\t")[0]), norm(l.split("\t")[1])) for l in lines]
pairs = [(e, s) for e, s in pairs if len(e.split()) <= MAXLEN - 2 and len(s.split()) <= MAXLEN - 2]
rng = np.random.default_rng(0); rng.shuffle(pairs)
pairs = pairs[:NUM_PAIRS]
for e, s in pairs[:5]: print(e, "  ->  ", s)

## 2. Tokenisation
`TextVectorization` layers map words to integer ids. The target vocabulary includes `<start>`/`<end>`; sequences are padded to a fixed length.

In [ ]:
src_texts = [e for e, _ in pairs]; tgt_texts = [f"<start> {s} <end>" for _, s in pairs]

src_vec = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAXLEN, standardize=None)
tgt_vec = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAXLEN + 1, standardize=None)
src_vec.adapt(src_texts); tgt_vec.adapt(tgt_texts)
src_vocab, tgt_vocab = src_vec.get_vocabulary(), tgt_vec.get_vocabulary()
src_len, tgt_len = len(src_vocab), len(tgt_vocab)
tgt_w2i = {w: i for i, w in enumerate(tgt_vocab)}
START, END = tgt_w2i["<start>"], tgt_w2i["<end>"]
print("source vocab:", src_len, " target vocab:", tgt_len)

X = src_vec(np.array(src_texts)).numpy()
T = tgt_vec(np.array(tgt_texts)).numpy()
dec_in, dec_out = T[:, :-1], T[:, 1:]                 # teacher forcing: input shifted by one
n_val = 2000
Xtr, Xva = X[:-n_val], X[-n_val:]; Dtr, Dva = dec_in[:-n_val], dec_in[-n_val:]; Otr, Ova = dec_out[:-n_val], dec_out[-n_val:]
print(Xtr.shape, Dtr.shape, Otr.shape)

## 3. Model with attention
The *same* layer objects are used to build both the training model and the step-by-step inference models, so the weights are shared.

In [ ]:
enc_emb   = layers.Embedding(src_len, EMB, mask_zero=True, name="enc_emb")
enc_rnn   = layers.Bidirectional(layers.GRU(UNITS, return_sequences=True, return_state=True), name="enc_bigru")
bridge    = layers.Dense(UNITS, activation="tanh", name="bridge")        # fwd+bwd final states -> decoder init state
dec_emb   = layers.Embedding(tgt_len, EMB, mask_zero=True, name="dec_emb")
dec_rnn   = layers.GRU(UNITS, return_sequences=True, return_state=True, name="dec_gru")
enc_proj  = layers.Dense(UNITS, name="enc_proj")                         # project BiGRU outputs (2*UNITS) to UNITS
attention = layers.Attention(name="attention")                           # dot-product (Luong) attention
combine   = layers.Dense(UNITS, activation="tanh", name="combine")
out_layer = layers.Dense(tgt_len, name="logits")

# ---- training graph ----
src_in = layers.Input((MAXLEN,), dtype="int32", name="src"); tgt_in = layers.Input((MAXLEN,), dtype="int32", name="tgt_in")
e = enc_emb(src_in)
enc_seq, f_state, b_state = enc_rnn(e)
init = bridge(layers.Concatenate()([f_state, b_state]))
enc_mem = enc_proj(enc_seq)
dec_seq, _ = dec_rnn(dec_emb(tgt_in), initial_state=init)
src_mask = enc_emb.compute_mask(src_in)
context = attention([dec_seq, enc_mem], mask=[None, src_mask])
logits = out_layer(combine(layers.Concatenate()([dec_seq, context])))
nmt = keras.Model([src_in, tgt_in], logits, name="nmt_attention")
nmt.summary()

## 4. Train
The loss ignores padding (`<pad>` has index 0).

In [ ]:
def masked_loss(y, logits):
    l = keras.losses.sparse_categorical_crossentropy(y, logits, from_logits=True)
    m = tf.cast(y != 0, l.dtype); return tf.reduce_sum(l * m) / tf.reduce_sum(m)
def masked_acc(y, logits):
    p = tf.argmax(logits, -1, output_type=y.dtype); m = tf.cast(y != 0, tf.float32)
    return tf.reduce_sum(tf.cast(p == y, tf.float32) * m) / tf.reduce_sum(m)

nmt.compile(optimizer=keras.optimizers.Adam(2e-3), loss=masked_loss, metrics=[masked_acc])
hist = nmt.fit([Xtr, Dtr], Otr, validation_data=([Xva, Dva], Ova), epochs=EPOCHS, batch_size=BATCH, verbose=2)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.history["masked_acc"], label="train"); ax[1].plot(hist.history["val_masked_acc"], label="val"); ax[1].set_title("Token accuracy"); ax[1].legend()
plt.show()

## 5. Inference models (encoder + one-step decoder with attention)

In [ ]:
# encoder model: source -> (encoder memory, initial decoder state)
encoder_model = keras.Model(src_in, [enc_mem, init])

# one-step decoder: (previous word, previous state, encoder memory) -> (logits, new state, attention weights)
w_in   = layers.Input((1,), dtype="int32"); s_in = layers.Input((UNITS,)); mem_in = layers.Input((MAXLEN, UNITS)); mask_in = layers.Input((MAXLEN,), dtype="bool")
d_seq, s_out = dec_rnn(dec_emb(w_in), initial_state=s_in)
ctx, scores = attention([d_seq, mem_in], mask=[None, mask_in], return_attention_scores=True)
step_logits = out_layer(combine(layers.Concatenate()([d_seq, ctx])))
decoder_step = keras.Model([w_in, s_in, mem_in, mask_in], [step_logits, s_out, scores])

def translate(sentence, return_attn=False):
    s = norm(sentence); x = src_vec(np.array([s])).numpy()
    mem, state = encoder_model.predict(x, verbose=0); mask = x != 0
    word, out, attn = np.array([[START]]), [], []
    for _ in range(MAXLEN):
        lg, state, sc = decoder_step.predict([word, state, mem, mask], verbose=0)
        nxt = int(lg[0, 0].argmax())
        if nxt == END: break
        out.append(tgt_vocab[nxt]); attn.append(sc[0, 0]); word = np.array([[nxt]])
    return (" ".join(out), s.split(), np.array(attn)) if return_attn else " ".join(out)

for s in ["i love you .", "where is the library ?", "how are you ?", "he is a good teacher .", "i do not know what to do .", "we are going to the beach tomorrow ."]:
    print(f"EN: {s:40s} ES: {translate(s)}")

## 6. Attention alignment heat-maps
Bright cells show which source (English) words the decoder attended to while producing each Spanish word.

In [ ]:
def plot_attention(sentence, ax):
    tr, src_toks, attn = translate(sentence, True)
    attn = attn[:, :len(src_toks)]
    ax.imshow(attn, cmap="viridis")
    ax.set_xticks(range(len(src_toks))); ax.set_xticklabels(src_toks, rotation=60)
    ax.set_yticks(range(len(tr.split()))); ax.set_yticklabels(tr.split()); ax.set_xlabel("source (English)"); ax.set_ylabel("output (Spanish)")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, s in zip(axes, ["i love you very much .", "he is a good teacher .", "where is the library ?"]): plot_attention(s, ax)
plt.tight_layout(); plt.show()

## 7. Quality: BLEU on held-out sentences

In [ ]:
from collections import Counter
def bleu(ref, hyp, maxn=4):
    ref, hyp = ref.split(), hyp.split()
    if not hyp: return 0.0
    logp = 0
    for n in range(1, maxn + 1):
        rc = Counter(tuple(ref[i:i+n]) for i in range(len(ref) - n + 1)); hc = Counter(tuple(hyp[i:i+n]) for i in range(len(hyp) - n + 1))
        match = sum(min(c, rc[g]) for g, c in hc.items()); tot = max(sum(hc.values()), 1)
        logp += np.log((match + 1e-9) / tot) / maxn
    bp = min(1.0, np.exp(1 - len(ref) / len(hyp)))
    return bp * np.exp(logp)

val_pairs = pairs[-n_val:][:200]
scores = [bleu(sp, translate(en)) for en, sp in val_pairs]
print(f"Average sentence-BLEU on {len(val_pairs)} held-out pairs: {100*np.mean(scores):.1f}")
for en, sp in val_pairs[:6]: print(f"\nEN : {en}\nREF: {sp}\nHYP: {translate(en)}")

## Conclusion
An encoder-decoder GRU with attention was trained for English -> Spanish translation. Instead of squeezing a whole sentence into one fixed-size vector, attention lets the decoder look back at the most relevant source words at each step - visible as the (roughly diagonal) alignment maps. Even on a 30k-pair subset the model produces fluent short translations; more data, more epochs, beam search and a Transformer would improve quality further.